# Stage 0. Data Harmonization and EDA

Notebook ini menyatukan dataset CP-AnemiC dan Eyes-Defy menjadi satu manifest terunifikasi, menurunkan label biner anemia memakai ambang sesuai populasi, lalu menampilkan eksplorasi distribusi hemoglobin, severity, dan situs.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from configs import paths
from src import data

print(paths.dataset_root())

## Build Unified Manifest

Membangun manifest terunifikasi dari kedua dataset. Setiap baris mewakili satu pasien dengan path citra region of interest, kadar hemoglobin, demografi, dan label biner anemia yang diturunkan memakai ambang sesuai populasi.

In [ ]:
manifest = data.build_manifest(save=False)
print("shape", manifest.shape)
manifest.head()

## Dataset Composition

Ringkasan jumlah sampel per dataset dan situs, keseimbangan kelas, distribusi severity, serta statistik hemoglobin dan umur.

In [ ]:
data.summarize(manifest)

## Hemoglobin Distribution

Distribusi kadar hemoglobin memperlihatkan perbedaan populasi. CP-AnemiC berasal dari anak dengan banyak kasus anemia, sedangkan Eyes-Defy berasal dari dewasa dengan mayoritas non-anemia. Garis putus-putus menandai ambang anak pada 11 g/dL.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, group in manifest.groupby("dataset"):
    ax.hist(group["hb_gdl"].dropna(), bins=30, alpha=0.6, label=name)
ax.axvline(11.0, color="black", linestyle="--", label="pediatric threshold")
ax.set_xlabel("Hemoglobin (g/dL)")
ax.set_ylabel("Count")
ax.set_title("Hemoglobin Distribution per Dataset")
ax.legend()
plt.tight_layout()
plt.show()

## Class Balance and Severity

Keseimbangan label anemia per dataset dan distribusi severity yang hanya tersedia pada CP-AnemiC.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

anemic_counts = manifest.groupby(["dataset", "anemic"]).size().unstack(fill_value=0)
anemic_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Anemia Label Balance per Dataset")
axes[0].set_xlabel("Dataset")
axes[0].set_ylabel("Count")

severity_order = ["Non-Anemic", "Mild", "Moderate", "Severe"]
cp_frame = manifest[manifest["dataset"] == "cp_anemic"]
severity_counts = cp_frame["severity"].value_counts().reindex(severity_order).fillna(0)
severity_counts.plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Severity Distribution on CP-AnemiC")
axes[1].set_xlabel("Severity")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Stratified Patient Split

Membagi data menjadi train, validation, dan test secara terstratifikasi per kombinasi dataset dan label anemik agar proporsi kelas terjaga. Setiap baris mewakili satu pasien sehingga pembagian ini setara dengan pembagian berbasis pasien.

In [ ]:
manifest = data.assign_stratified_split(manifest)
split_counts = manifest.groupby(["dataset", "split"]).size().unstack(fill_value=0)
print(split_counts.to_string())

## Save Manifest

Manifest final beserta kolom split disimpan ke folder outputs untuk dipakai stage berikutnya.

In [ ]:
output_path = paths.OUTPUTS / "manifest.csv"
manifest.to_csv(output_path, index=False)
print("manifest saved to", output_path)